# ST-OMR Meter V5-3H — Authoritative Rescue TRAIN Execution

Single-run fail-closed wrapper pinned to V5-3G exact CI-green HEAD `b36a9d2f5daade2c3568cac8cbc736ca75ca435f`. It materializes TRAIN-only frozen-negative features and runs the already-fixed rescue recipe once, then stops before TRAIN acceptance, Historical Validation, First-30, V5 VAL, and FINAL_HOLDOUT.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import time

EXPECTED_HEAD = "b36a9d2f5daade2c3568cac8cbc736ca75ca435f"
EXPECTED_CI_RUN_ID = 32769348282
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = (
    MYDRIVE
    / "ST-OMR-D10"
    / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
)
for name, path in {
    "DATA_ROOT": DATA_ROOT,
    "CHECKPOINT_ROOT": CHECKPOINT_ROOT,
    "M4A_ROOT": M4A_ROOT,
    "D10_ROOT": D10_ROOT,
}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE CHECK = PASS")

if not REPO.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"REPO git repository degil: {REPO}")
remotes = subprocess.check_output(
    ["git", "-C", str(REPO), "remote"], text=True
).split()
if "origin" not in remotes:
    subprocess.check_call(
        ["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL]
    )
else:
    subprocess.check_call(
        ["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL]
    )
subprocess.check_call(
    ["git", "-C", str(REPO), "fetch", "origin", EXPECTED_HEAD, "--depth", "1"]
)
fetched_head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if fetched_head != EXPECTED_HEAD:
    raise RuntimeError(
        f"FETCH_HEAD mismatch: expected={EXPECTED_HEAD} actual={fetched_head}"
    )
subprocess.check_call(
    ["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD]
)
actual_head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
if actual_head != EXPECTED_HEAD:
    raise RuntimeError(
        f"HEAD mismatch: expected={EXPECTED_HEAD} actual={actual_head}"
    )
if subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"], text=True
).strip():
    raise RuntimeError("Repository worktree temiz degil")
print("REPOSITORY CHECK = PASS")
print("HEAD =", actual_head)
print("CI RUN ID =", EXPECTED_CI_RUN_ID)

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")]
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--index-url",
        "https://download.pytorch.org/whl/cpu",
        "-r",
        str(REPO / "requirements-training.txt"),
    ]
)
expected_runtime = {
    "lxml": "6.1.1",
    "verovio": "6.2.1",
    "CairoSVG": "2.8.2",
    "Pillow": "12.3.0",
    "scipy": "1.18.0",
    "torch": "2.13.0+cpu",
}
for package, expected in expected_runtime.items():
    actual = metadata.version(package)
    if actual != expected:
        raise RuntimeError(
            f"Runtime mismatch {package}: expected={expected} actual={actual}"
        )
subprocess.check_call([sys.executable, "-m", "pip", "check"])
print("PINNED RUNTIME = PASS")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from st_omr_training import meter_v5_1_bbox_pilot as v51
from st_omr_training import meter_v5_2b_specialist_adaptation as v52b
from st_omr_training import meter_v5_3e_rescue_training_preregistration_v1 as v53e
from st_omr_training import meter_v5_3g_authoritative_rescue_training_v1 as rescue

if rescue.V53F_HEAD_SHA != "7ed41f2872058ac5e3e52df756b9098a1d60052d":
    raise RuntimeError("V5-3F prerequisite binding changed")
if rescue.prerequisite_contract()["v5_3f_head_sha"] != rescue.V53F_HEAD_SHA:
    raise RuntimeError("V5-3G prerequisite contract mismatch")
print("MODULE IMPORT/BINDING = PASS")

DIGIT2_FROZEN = v52b.locate_checkpoint_by_sha_v1(
    CHECKPOINT_ROOT, v52b.DIGIT2_SHA256
)
DIGIT3_FROZEN = v52b.locate_checkpoint_by_sha_v1(
    CHECKPOINT_ROOT, v52b.DIGIT3_SHA256
)
print("FROZEN CHECKPOINTS = PASS")
print("2-AI =", DIGIT2_FROZEN)
print("3-AI =", DIGIT3_FROZEN)

ANN_DIR = DATA_ROOT / v51.ANNOTATIONS_DIR
REPORT_PATH = ANN_DIR / rescue.REPORT_NAME
ARTIFACT_DIR = ANN_DIR / rescue.ARTIFACT_DIR_NAME
TEMP_ARTIFACT_DIR = ANN_DIR / rescue.TEMP_ARTIFACT_DIR_NAME
ENVELOPE_PATH = ANN_DIR / f"v5_3g_execution_envelope_{EXPECTED_HEAD}.json"
for path in (REPORT_PATH, ARTIFACT_DIR, TEMP_ARTIFACT_DIR, ENVELOPE_PATH):
    if path.exists():
        raise RuntimeError(f"Refusing overwrite/rerun: {path}")
print("OUTPUT GUARD = PASS")

required_safety = {
    "single_authoritative_train_entry": True,
    "data_surfaces": ("v5_train", "historical_train"),
    "frozen_feature_extractor_reused": "v5-2n",
    "new_bbox": False,
    "new_crop_geometry": False,
    "new_spatial_heuristic": False,
    "frozen_checkpoint_read": True,
    "frozen_checkpoint_write": False,
    "frozen_model_mutation_allowed": False,
    "trainable_surface": "new-rescue-parameters-only",
    "digit4_loaded": False,
    "digit4_frozen": True,
    "rescue_artifact_write": True,
    "original_specialist_checkpoint_replacement": False,
    "threshold_tuning": False,
    "hyperparameter_sweep": False,
    "automatic_second_configuration": False,
    "historical_validation_opened": False,
    "first30_opened": False,
    "v5_reserve_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "resolver_wiring": False,
    "production_promotion": False,
    "colab_execution_wrapper_present": False,
}
boundary = rescue.safety_boundary()
for key, expected in required_safety.items():
    if boundary.get(key) != expected:
        raise RuntimeError(
            f"Safety boundary mismatch: {key}={boundary.get(key)!r}"
        )
contract = rescue.execution_contract()
if contract.get("one_shot_non_overwriting") is not True:
    raise RuntimeError("One-shot execution guard missing")
if contract.get("exact_sha_colab_wrapper_required_for_external_execution") is not True:
    raise RuntimeError("Exact-SHA wrapper requirement missing")
if contract["authoritative_group_counts"] != v53e.EXPECTED_TRAIN_GROUP_COUNTS:
    raise RuntimeError("Authoritative group-count contract changed")
print("SAFETY BOUNDARY = PASS")
print("TRAIN ONLY = PASS | 4-AI=FROZEN")
print("HISTORICAL_VALIDATION=CLOSED | FIRST-30=CLOSED | V5_VAL=CLOSED")
print("FINAL_HOLDOUT=LOCKED | PRODUCTION_PROMOTION=FALSE")

started = time.time()
def progress(processed, total, phase):
    if processed == total or processed % 2048 == 0:
        print(
            phase,
            f"{processed}/{total}",
            f"| elapsed={int(time.time() - started)}s",
        )

report = rescue.run_authoritative_rescue_training_v1(
    DATA_ROOT,
    m4a_root=M4A_ROOT,
    d10_root=D10_ROOT,
    digit2_frozen=DIGIT2_FROZEN,
    digit3_frozen=DIGIT3_FROZEN,
    confirmation=rescue.APPROVAL_TOKEN,
    progress=progress,
)

if not REPORT_PATH.is_file():
    raise RuntimeError(f"Authoritative report not written: {REPORT_PATH}")
saved_report_bytes = REPORT_PATH.read_bytes()
saved_report = json.loads(saved_report_bytes.decode("utf-8"))
if saved_report != report:
    raise RuntimeError("Saved report mismatch")
if report.get("single_authoritative_execution_completed") is not True:
    raise RuntimeError("Single authoritative execution receipt missing")
if report.get("candidate_configuration_count") != 1:
    raise RuntimeError("Candidate configuration count changed")
if report.get("numerical_integrity_gate", {}).get("gate") != "PASS":
    raise RuntimeError("Numerical integrity gate did not PASS")
if report.get("frozen_state_isolation_gate", {}).get("gate") != "PASS":
    raise RuntimeError("Frozen-state isolation gate did not PASS")
if report.get("train_performance_gate_executed") is not False:
    raise RuntimeError("TRAIN performance gate opened unexpectedly")
if report.get("historical_validation_retention_executed") is not False:
    raise RuntimeError("Historical validation opened unexpectedly")
if report.get("first30_opened") is not False:
    raise RuntimeError("First-30 opened unexpectedly")
if report.get("v5_validation_opened") is not False:
    raise RuntimeError("V5 validation opened unexpectedly")
if report.get("final_holdout_locked") is not True:
    raise RuntimeError("FINAL_HOLDOUT lock changed")
if report.get("runtime_authority_changed") is not False:
    raise RuntimeError("Runtime authority changed unexpectedly")
if report.get("production_promotion") is not False:
    raise RuntimeError("Production promotion changed unexpectedly")

artifact_sha256 = {}
group_fingerprints = {}
for digit in ("2", "3"):
    item = report["per_specialist"][digit]
    materialization = item["materialization"]
    execution = item["execution"]
    expected_counts = v53e.EXPECTED_TRAIN_GROUP_COUNTS[digit]
    if materialization.get("group_counts") != expected_counts:
        raise RuntimeError(f"{digit}-AI group count mismatch")
    fingerprints = materialization.get("group_fingerprints")
    if not isinstance(fingerprints, dict) or set(fingerprints) != set(v53e.TRAIN_GROUPS):
        raise RuntimeError(f"{digit}-AI group fingerprint surface mismatch")
    if not all(
        isinstance(value, str) and len(value) == 64
        for value in fingerprints.values()
    ):
        raise RuntimeError(f"{digit}-AI invalid group fingerprint")
    group_fingerprints[digit] = fingerprints
    if item.get("frozen_state_bit_identical") is not True:
        raise RuntimeError(f"{digit}-AI frozen-state isolation missing")
    if item.get("frozen_state_before") != item.get("frozen_state_after"):
        raise RuntimeError(f"{digit}-AI frozen state changed")
    if execution.get("authoritative_dataset_execution") is not True:
        raise RuntimeError(f"{digit}-AI execution not marked authoritative")
    if execution.get("optimizer_steps") != v53e.FIXED_OPTIMIZER_STEPS:
        raise RuntimeError(f"{digit}-AI optimizer step count changed")
    for key in (
        "finite_initial_parameters",
        "finite_losses",
        "finite_gradients",
        "finite_post_step_parameters",
    ):
        if execution.get(key) is not True:
            raise RuntimeError(f"{digit}-AI numerical guard failed: {key}")
    if execution.get("checkpoint_write") is not False:
        raise RuntimeError(f"{digit}-AI V5-3F checkpoint path opened")
    if execution.get("protected_evaluation_opened") is not False:
        raise RuntimeError(f"{digit}-AI protected evaluation opened")
    artifact = item["artifact"]
    artifact_path = Path(artifact["artifact_path"])
    if not artifact_path.is_file() or artifact.get("reload_verified") is not True:
        raise RuntimeError(f"{digit}-AI rescue artifact missing/unverified")
    actual_sha = v52b._sha_file(artifact_path)
    if actual_sha != artifact.get("artifact_sha256"):
        raise RuntimeError(f"{digit}-AI rescue artifact SHA mismatch")
    artifact_sha256[digit] = actual_sha

post_run_head = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
if post_run_head != EXPECTED_HEAD:
    raise RuntimeError(
        f"Post-run HEAD mismatch: expected={EXPECTED_HEAD} actual={post_run_head}"
    )
if subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain"], text=True
).strip():
    raise RuntimeError("Post-run repository worktree temiz degil")

envelope = {
    "schema": "st-omr-meter-v5-3h-authoritative-rescue-execution-envelope-v1",
    "repository": REPOSITORY,
    "expected_head": EXPECTED_HEAD,
    "actual_head": post_run_head,
    "ci_run_id": EXPECTED_CI_RUN_ID,
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "report_path": str(REPORT_PATH),
    "report_sha256": hashlib.sha256(saved_report_bytes).hexdigest(),
    "artifact_sha256": artifact_sha256,
    "group_fingerprints": group_fingerprints,
    "single_authoritative_execution_completed": True,
    "candidate_configuration_count": 1,
    "numerical_integrity_gate": "PASS",
    "frozen_state_isolation_gate": "PASS",
    "train_performance_gate_executed": False,
    "historical_validation_opened": False,
    "first30_opened": False,
    "v5_reserve_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "digit4_frozen": True,
    "threshold_tuning": False,
    "hyperparameter_sweep": False,
    "automatic_second_configuration": False,
    "runtime_authority_changed": False,
    "production_promotion": False,
}
v51._atomic_write_json(ENVELOPE_PATH, envelope)
if not ENVELOPE_PATH.is_file():
    raise RuntimeError("V5-3H execution envelope not written")
print("V5-3G AUTHORITATIVE TRAIN EXECUTION = PASS")
print("NUMERICAL INTEGRITY = PASS")
print("FROZEN STATE ISOLATION = PASS")
print("TRAIN PERFORMANCE GATE = CLOSED")
print("HISTORICAL VALIDATION = CLOSED")
print("FIRST-30 = CLOSED")
print("V5 VAL = CLOSED")
print("FINAL_HOLDOUT = LOCKED")
print("REPORT =", REPORT_PATH)
print("ENVELOPE =", ENVELOPE_PATH)
print("ARTIFACT SHA256 =", json.dumps(artifact_sha256, sort_keys=True))
